# Лабораторная работа № 4
## Условная вероятность и формула полной вероятности

**Тема:** классическое решение и моделирование задачи с двумя урнами.

**Цель:** вычислить вероятность по формуле полной вероятности и проверить её методом Монте-Карло.

**Условие.** В первой урне 15 белых и 5 чёрных шаров, во второй — 6 белых и 1 чёрный. Из первой урны случайно вынимают 3 шара **без возвращения** и перекладывают во вторую. Затем из второй урны случайно вынимают 1 шар. Найти вероятность того, что последний шар белый.

### 1. Классическое решение

Пусть $A$ — событие «из второй урны вынут белый шар»; $H_k$ — гипотеза «из первой во вторую урну перенесено ровно $k$ белых шаров», $k=0,1,2,3$.

Всего способов выбрать 3 шара из первой урны: $C_{20}^{3}=1140$.

Вероятность гипотезы $H_k$ (гипергеометрическое распределение):

$$P(H_k)=\frac{C_{15}^{k}C_{5}^{3-k}}{C_{20}^{3}}.$$

После переноса во второй урне будет 10 шаров, среди которых $6+k$ белых. Поэтому

$$P(A\mid H_k)=\frac{6+k}{10}.$$

По **формуле полной вероятности**:

$$P(A)=\sum_{k=0}^{3}P(H_k)P(A\mid H_k).$$

In [1]:
from math import comb
from fractions import Fraction

ways_total = comb(20, 3)
probability = Fraction(0, 1)

print('Перенесено белых | Число способов | P(H_k) | P(A | H_k)')
for k in range(4):
    ways = comb(15, k) * comb(5, 3-k)
    p_h = Fraction(ways, ways_total)
    p_white_given_h = Fraction(6+k, 10)
    probability += p_h * p_white_given_h
    print(f'{k:^17} | {ways:^13} | {str(p_h):^8} | {p_white_given_h}')

print(f'\nТочная вероятность: {probability} = {float(probability):.6f}')
assert probability == Fraction(33, 40)


Перенесено белых | Число способов | P(H_k) | P(A | H_k)
        0         |      10       |  1/114   | 3/5
        1         |      150      |   5/38   | 7/10
        2         |      525      |  35/76   | 4/5
        3         |      455      |  91/228  | 9/10

Точная вероятность: 33/40 = 0.825000


Подставим все четыре гипотезы:

$$P(A)=\frac{10}{1140}\cdot\frac6{10}+\frac{150}{1140}\cdot\frac7{10}+\frac{525}{1140}\cdot\frac8{10}+\frac{455}{1140}\cdot\frac9{10}$$

$$=\frac{60+1050+4200+4095}{11400}=\frac{9405}{11400}=\frac{33}{40}=0{,}825.$$

**Ответ по классической формуле: $P(A)=0{,}825=82{,}5\%$.**

### 2. Проверка классического решения другим способом

После переноса во второй урне 10 шаров. Из них 7 первоначально находились во второй урне и 3 были перенесены из первой. Если вытянут шар из первоначальной части, шанс белого $6/7$; если перенесённый — $15/20$.

$$P(A)=\frac7{10}\cdot\frac67+\frac3{10}\cdot\frac{15}{20}=\frac6{10}+\frac9{40}=\frac{33}{40}.$$

In [2]:
check = Fraction(7, 10)*Fraction(6, 7) + Fraction(3, 10)*Fraction(15, 20)
print('Аналитическая проверка:', check, '=', float(check))
assert check == probability


Аналитическая проверка: 33/40 = 0.825


### 3. Моделирование методом Монте-Карло

Многократно воспроизводим условие задачи. В каждом испытании заново создаём обе урны, случайно перекладываем **три разных** шара из первой во вторую и выбираем один шар из второй. Отношение числа белых шаров к количеству испытаний даёт экспериментальную оценку вероятности.

Используем фиксированное начальное состояние генератора, чтобы результат воспроизводился. Сравним оценку после 100, 1000, 10 000 и 100 000 испытаний.

In [3]:
import random

rng = random.Random(20260928)
checkpoints = (100, 1_000, 10_000, 100_000)
white_outcomes = 0
estimates = []

for experiment in range(1, checkpoints[-1] + 1):
    first_urn = ['белый'] * 15 + ['чёрный'] * 5
    second_urn = ['белый'] * 6 + ['чёрный']

    transferred = rng.sample(first_urn, 3)  # без возвращения
    second_urn.extend(transferred)
    drawn = rng.choice(second_urn)
    if drawn == 'белый':
        white_outcomes += 1

    if experiment in checkpoints:
        estimate = white_outcomes / experiment
        estimates.append(estimate)
        print(f'{experiment:>6} испытаний: P ≈ {estimate:.5f}; '
              f'отклонение = {abs(estimate - float(probability)):.5f}')

assert abs(estimates[-1] - float(probability)) < 0.01


   100 испытаний: P ≈ 0.78000; отклонение = 0.04500
  1000 испытаний: P ≈ 0.83000; отклонение = 0.00500
 10000 испытаний: P ≈ 0.82110; отклонение = 0.00390
100000 испытаний: P ≈ 0.82404; отклонение = 0.00096


### 4. Сравнение результатов и вывод

**Теоретическая вероятность:** $33/40=0{,}825$.

**Модельная вероятность:** при 100 000 испытаний получено $0{,}82404$; при другом начальном состоянии генератора оценка немного изменится.

**Вывод:** классический расчёт и компьютерное моделирование согласуются. При большом числе независимых повторений относительная частота белого шара близка к вероятности $0{,}825$.